# 🎬 ViniVideo AI — Kaggle Qualidade (Wan 2.1)

Este notebook é o modo **Qualidade** do ViniVideo AI.

Ele usa **Wan 2.1 T2V 1.3B**, gera tomadas curtas em resolução próxima de 480p e depois converte o resultado para **24 fps**, mirando uma aparência bem mais estável e cinematográfica que o modo Turbo.

## Antes de rodar

1. Kaggle → **Settings → Accelerator → GPU T4 x2**.
2. Deixe **Internet** ativada.
3. Use **Run All**.
4. Comece com **5 segundos**. Depois teste 10 segundos.

> Importante: este modelo não é o Gemini Omni 1.1 Flash. Ele é uma alternativa aberta para priorizar qualidade dentro da GPU gratuita do Kaggle.


In [ ]:
import os
import torch

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "YES"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "Ative Settings → Accelerator → GPU T4 x2 no Kaggle."
    )

print("GPU 0:", torch.cuda.get_device_name(0))
print("Quantidade de GPUs:", torch.cuda.device_count())


In [ ]:
!pip -q install -U "diffusers>=0.35,<0.41" "transformers>=4.48,<4.58" "accelerate>=1.2,<2" "huggingface-hub>=0.30" safetensors ftfy imageio imageio-ffmpeg hf-xet sentencepiece
print("Dependências de qualidade prontas ✓")


In [ ]:
import gc
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from transformers import UMT5EncoderModel

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print("🎬 Carregando Wan 2.1 1.3B...")
print("O primeiro carregamento é mais demorado que o modo Turbo.")

text_encoder = UMT5EncoderModel.from_pretrained(
    MODEL_ID,
    subfolder="text_encoder",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

vae = AutoencoderKLWan.from_pretrained(
    MODEL_ID,
    subfolder="vae",
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)

pipe = WanPipeline.from_pretrained(
    MODEL_ID,
    text_encoder=text_encoder,
    vae=vae,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config,
    flow_shift=3.0,
)

pipe.enable_model_cpu_offload(gpu_id=0)

print("✅ Wan 2.1 pronto")


In [ ]:
# ===== CONFIGURAÇÃO =====
PROMPT = """A charming blue macaw in a colorful tropical neighborhood at golden hour,
high-quality 3D animated feature-film look, expressive eyes, detailed feathers,
natural beak proportions, warm cinematic sunlight, soft depth of field.
A small friendly green creature balances on the macaw's head.
The camera begins in a close-up and slowly pulls backward while the macaw reacts,
blinks and gestures naturally. Gentle wind moves leaves through the scene.
Maintain the same character identity, body proportions, colors and environment
throughout the entire shot. Smooth coherent motion, no random cuts."""

NEGATIVE_PROMPT = """low quality, worst quality, blurry, flicker, jitter,
warped face, changing character design, duplicated body parts, extra wings,
deformed beak, broken anatomy, random cuts, camera teleport, text, watermark,
still image, frozen subject, oversaturated, noisy, compression artifacts"""

DURATION_SECONDS = 5
ASPECT = "9:16"   # "9:16" ou "16:9"
SEED = 20261007

# 24–30 steps = bom equilíbrio na T4.
STEPS = 28
GUIDANCE = 5.0

print("Prompt configurado ✓")
print("Duração:", DURATION_SECONDS, "s")
print("Formato:", ASPECT)


In [ ]:
from pathlib import Path
from diffusers.utils import export_to_video
from IPython.display import Video, display, FileLink
import subprocess
import torch
import gc
import math

OUTPUT_DIR = Path("/kaggle/working/ViniVideoAI_Quality")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def dimensions(aspect):
    if aspect == "9:16":
        # Próximo de 480p vertical, divisível por 32.
        return 448, 800
    return 800, 448

width, height = dimensions(ASPECT)

# Wan trabalha muito bem com 81 frames (~5 s a 16 fps).
scene_seconds = 5
scene_count = max(1, math.ceil(int(DURATION_SECONDS) / scene_seconds))
scene_paths = []

print()
print(f"🎞️ Gerando {scene_count} tomada(s) em {width}x{height}...")

for scene_index in range(scene_count):
    start = scene_index * scene_seconds
    remaining = int(DURATION_SECONDS) - start
    this_duration = max(1, min(scene_seconds, remaining))

    scene_path = OUTPUT_DIR / f"wan_scene_{scene_index + 1:03d}.mp4"

    if scene_path.exists() and scene_path.stat().st_size > 2048:
        print(f"♻️ Tomada {scene_index + 1} já existe; reutilizando.")
        scene_paths.append(scene_path)
        continue

    continuity = (
        " Continue the exact same visual identity, lighting, palette, "
        "character proportions and camera language from the previous shot."
        if scene_index > 0 else ""
    )

    scene_prompt = (
        PROMPT
        + continuity
        + f" This is shot {scene_index + 1} of {scene_count}."
    )

    generator = torch.Generator(device="cpu").manual_seed(
        int(SEED) + scene_index
    )

    print(
        f"🎬 Tomada {scene_index + 1}/{scene_count} "
        f"• {this_duration}s • 81 frames • {STEPS} steps"
    )

    def progress(pipe, step, timestep, kwargs):
        pct = int(((step + 1) / max(1, STEPS)) * 100)
        if pct % 10 == 0 or step == 0 or step + 1 == STEPS:
            print(f"   Difusão: {pct}%")
        return kwargs

    result = pipe(
        prompt=scene_prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=height,
        width=width,
        num_frames=81,
        num_inference_steps=int(STEPS),
        guidance_scale=float(GUIDANCE),
        generator=generator,
        callback_on_step_end=progress,
    )

    frames = result.frames[0]

    export_to_video(
        frames,
        str(scene_path),
        fps=16,
    )

    scene_paths.append(scene_path)

    del result, frames
    gc.collect()
    torch.cuda.empty_cache()

    print(f"✅ Tomada {scene_index + 1} salva")

concat_file = OUTPUT_DIR / "concat.txt"
with open(concat_file, "w", encoding="utf-8") as f:
    for path in scene_paths:
        safe = str(path).replace("'", "'\\''")
        f.write(f"file '{safe}'\n")

raw_path = OUTPUT_DIR / "ViniVideoAI_Wan_raw.mp4"

subprocess.run(
    [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(concat_file),
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "17",
        "-pix_fmt", "yuv420p",
        str(raw_path),
    ],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

final_path = OUTPUT_DIR / "ViniVideoAI_Wan_24fps.mp4"

# Aproxima a entrega do vídeo de referência: 24 fps e saída vertical limpa.
vf = (
    "minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:"
    "me_mode=bidir:vsbmc=1,"
    "scale=360:640:force_original_aspect_ratio=increase,"
    "crop=360:640"
    if ASPECT == "9:16"
    else
    "minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:"
    "me_mode=bidir:vsbmc=1,"
    "scale=640:360:force_original_aspect_ratio=increase,"
    "crop=640:360"
)

subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(raw_path),
        "-vf", vf,
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "17",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(final_path),
    ],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

print()
print("✅ VÍDEO QUALIDADE PRONTO")
print("Arquivo:", final_path)

display(Video(str(final_path), embed=True))
display(
    FileLink(
        str(final_path),
        result_html_prefix="⬇️ Baixar MP4 qualidade: "
    )
)


## O que este modo melhora

- Modelo de vídeo real **Wan 2.1 1.3B**, em vez de AnimateDiff.
- Geração em resolução próxima de **480p**.
- **81 frames por tomada**.
- Mais passos de difusão para melhorar forma, textura e movimento.
- Saída final em **24 fps**, como o vídeo de referência.
- Checkpoint por tomada: se a célula final for executada de novo na mesma sessão, tomadas já concluídas podem ser reutilizadas.

### Limitação atual

O Wan 1.3B deste notebook é **texto→vídeo**. Ele melhora bastante a qualidade visual, mas ainda não tem o mesmo controle multimodal e áudio nativo do Gemini Omni 1.1 Flash.

A próxima etapa para consistência de personagem por imagem é integrar **Wan VACE 1.3B** ou um pipeline de referência.
